# 4. Análisis univariado

Se describe cada una de las 17 predictoras por separado: su forma, su asimetría y sus valores atípicos.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

## 4.1 Variables numéricas

In [ ]:
desc = df[NUMERIC].describe().T
desc["asimetria"] = df[NUMERIC].skew()
desc["% atipicos (IQR)"] = [iqr_outliers(df[c]) for c in NUMERIC]
desc.round(2)

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(12, 15))
for ax, col in zip(axes.flat, NUMERIC):
    data = df[col].dropna()
    log = col in ("area", "dist_to_road")
    if log:
        data = np.log10(data + 1)
    ax.hist(data, bins=50, color=CLASS_COLORS["Alta"], edgecolor=SURFACE, linewidth=0.4)
    ax.set_title(f"log10({col} + 1)" if log else col, fontsize=10)
    ax.set_ylabel("Plantas")
    ax.axvline(data.median(), color=NEUTRAL, lw=1.2, ls="--")
for ax in axes.flat[len(NUMERIC):]:
    ax.remove()
fig.suptitle("Distribución de las predictoras numéricas (línea discontinua = mediana)",
             x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Lectura de las distribuciones**

- **`area` y `dist_to_road`** son las variables más asimétricas (asimetría de 72.6 y 48.7). En escala original son ilegibles; en escala logarítmica muestran una forma aproximadamente unimodal. Esto justifica una **transformación `log1p`** antes de escalar.
- **`longitude`** es multimodal, con picos en Norteamérica (−120° a −75°), Europa (0–30°) y Asia (75–145°). Refleja dónde se concentran las plantas, no una propiedad física (ver capítulo 5).
- **`latitude`** tiene asimetría negativa: solo el 4.6 % de las plantas está en el hemisferio sur.
- **`slope`** (asimetría 4.0) y **`elevation`** (3.2) se concentran cerca de cero, con colas largas: la mayoría de plantas está en terrenos bajos y planos.
- **`curvature`** tiene el mayor porcentaje de atípicos por IQR (18.95 %), pero porque su rango intercuartílico es casi nulo, no por errores. Hay 10 232 valores exactamente iguales a 0.
- **`aspect`** y **`wind_direction`** son **variables circulares** (0° = 360°). Usadas tal cual, un modelo considera que 359° y 1° están lejos. Se recomienda codificarlas como **seno y coseno**.
- **`ambient_temperature`**, **`ghi`** y **`humidity`** son las más cercanas a una forma acampanada.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, col in zip(axes, ["elevation", "slope", "wind_speed", "humidity"]):
    ax.boxplot(df[col].dropna(), vert=False, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#cde2fb", edgecolor=CLASS_COLORS["Alta"]),
               medianprops=dict(color="#0b0b0b"),
               flierprops=dict(marker="o", markersize=2, alpha=0.3,
                               markeredgecolor=NEUTRAL))
    ax.set_title(col, fontsize=10); ax.set_yticks([])
fig.suptitle("Valores atípicos en variables con cola larga", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 4.2 Variables categóricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, CATEGORICAL):
    vc = df[col].value_counts().sort_values()
    ax.barh(vc.index, vc.values, height=0.6, color=CLASS_COLORS["Alta"])
    for y, v in enumerate(vc.values):
        ax.text(v, y, f" {100 * v / len(df):.1f} %", va="center", fontsize=8.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlim(0, vc.max() * 1.22)
    ax.grid(axis="y", visible=False)
fig.suptitle("Frecuencia de las predictoras categóricas", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

- **`slope_type`:** el 79.3 % de las plantas está en terreno *plano o casi plano*. La categoría *Escarpado o abrupto* tiene solo 4 casos; en el one-hot encoding produce una columna casi vacía y puede agruparse con *Muy fuerte*.
- **`curvature_type`:** el 81.1 % está en superficies planas o intermedias.
- **`aspect_type`:** es la más equilibrada, con las ocho orientaciones entre el 8 % y el 14 %. *Flat* (1.9 %) corresponde a `aspect = −1`.
- **`dt_wind`:** predominan los vientos del suroeste (35.7 %). Norte, noreste y noroeste suman menos del 1 %, otro caso de niveles raros.